# Ch10. Dynamic Regression
**Forecasting: Principles & Practice (Python Edition)**  
Lab Notebook · [github.com/bcseong2/fpppy-labs](https://github.com/bcseong2/fpppy-labs)

In [ ]:
%pip install statsforecast neuralforecast hierarchicalforecast mlforecast utilsforecast

## [Slide 5] StatsForecast Implementation

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, ARIMA

# Automatic ARIMA order selection for errors
sf = StatsForecast(
    models=[AutoARIMA(season_length=1)],
    freq='QS'
)
sf.fit(df=y_df, X_df=x_df)

## [Slide 8] US Consumption: Code

In [ ]:
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from statsforecast.utils import AirPassengersDF

us_change = pd.read_csv("data/US_change.csv", parse_dates=["ds"])
y_df = us_change[["unique_id", "ds", "Consumption"]]
x_df = us_change[["unique_id", "ds", "Income"]]

sf = StatsForecast(
    models=[AutoARIMA(season_length=1)],
    freq='QS'
)
sf.fit(df=y_df.rename(columns={"Consumption": "y"}),
       X_df=x_df.rename(columns={"Income": "x"}))

summary = sf.fitted_[0, 0].summary()
print(summary)

## [Slide 12] Forecasting: US Consumption

In [ ]:
from utilsforecast.processing import make_future_dataframe

# Assumption: future income changes = historical mean
future_predictors = (
    make_future_dataframe(uids, last_times, freq="QS", h=8)
    .assign(Income=us_change["Income"].mean())
)

fc = sf.forecast(
    df=us_change, h=8,
    X_df=future_predictors,
    target_col="Consumption",
    level=[80, 95]
)

## [Slide 14] Forecasting: Electricity Demand

In [ ]:
# 14-day ahead forecast with constant 26°C temperature
future_elec = pd.DataFrame({
    "unique_id": "VIC",
    "ds": pd.date_range("2015-01-01", periods=14, freq="D"),
    "Temperature": 26.0,
    "Temperature_sq": 26.0**2,
    "Weekday": [1,1,1,1,1,0,0,  # Mon-Sun
                1,1,1,1,1,0,0],
})

fc_elec = sf.forecast(
    df=elec_df, h=14,
    X_df=future_elec,
    level=[80, 95]
)

## [Slide 20] Example: Australian Café/Restaurant Expenditure

In [ ]:
from utilsforecast.feature_engineering import fourier, pipeline
from functools import partial

# Test K = 1, 2, ..., 6
for k in range(1, 7):
    features = [partial(fourier, season_length=12, k=k)]
    aus_cafe_ft, fut_ft = pipeline(
        aus_cafe, features=features, freq="MS", h=24
    )
    sf = StatsForecast(
        models=[AutoARIMA(season_length=1, d=0)],
        freq='MS'
    )
    sf.fit(df=aus_cafe_ft)
    print(f"K={k}: AICc = {sf.fitted_[0,0].model_['ic']:.2f}")

## [Slide 26] Insurance Example: Code

In [ ]:
# Create lagged predictor columns
fit_df = insurance.assign(**{
    f"TVadverts_{lag}": insurance["TVadverts"].shift(lag)
    for lag in (1, 2, 3)
}).dropna()

sf = StatsForecast(
    models=[AutoARIMA(season_length=1)],
    freq='MS'
)
sf.fit(
    df=fit_df[["unique_id","ds","Quotes"]].rename(
        columns={"Quotes": "y"}),
    X_df=fit_df[["unique_id","ds","TVadverts","TVadverts_1"]]
)

# Forecast 12 months with advertising = 8 units
future_adv = pd.DataFrame({
    "unique_id": "insurance",
    "ds": pd.date_range("2005-05-01", periods=12, freq="MS"),
    "TVadverts": 8.0,
    "TVadverts_1": [insurance["TVadverts"].iloc[-1]] + [8.0]*11,
})
fc = sf.forecast(h=12, X_df=future_adv, level=[80, 95])